# h5py code improvement
At one point, I believed I needed to generate h5 files for the entire SGA 2025 catalog. I attempted on the original code, but it estimated 17 hours for completion, so I used Claude to see if it could improve the speed.

#### Results:
Claude estimated a 15 minute run time for the best setup, and when I used it, it took 2 hours (still much better!)

I/O benchmarking harness for the H5 generator (NERSC / Lustre).

Tests read throughput under different executor types and worker counts,
using a stratified sample of the *real* catalog so results reflect the
actual file layout / compression / anchor_source mix on /pscratch,
rather than a synthetic benchmark that won't transfer to the real run.

Usage (in a notebook cell, after you've built `targets` via format_data):

    from benchmark_harness import run_benchmark
    results_df = run_benchmark(
        targets,
        npix=152,
        old_anchors=True,
        sample_size=500,
    )
    results_df.sort_values("galaxies_per_sec", ascending=False)

CAVEAT ON CACHING: every config re-reads files. After the first pass,
some files may be warm in the OS/Lustre client cache, which can inflate
throughput for later configs. This harness draws a fresh random
sub-sample per run (different `seed`) so you can sanity check whether
cache warmth is skewing results -- run it twice with different seeds
and see if the ranking of configs holds up. For a fully rigorous
comparison, increase sample_size until config order stops mattering.


In [1]:
import time
import numpy as np
import pandas as pd
import fitsio
from astropy.io import fits
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

In [2]:
def get_band(bands, priority):
    for band in priority:
        if band in bands:
            return bands[band]
    return None
 
 
def read_one_bench(args):
    """
    Standalone (picklable) version of read_one, instrumented with
    per-stage timings. Takes a plain tuple so it works unmodified with
    both ThreadPoolExecutor and ProcessPoolExecutor.
    """
    i, path_str, anchor_source, old_anchors, npix, return_nbytes = args
    path = Path(path_str)
    t_start = time.perf_counter()
    timings = {"open": 0.0, "read": 0.0, "stack": 0.0}
    img_data = None
 
    try:
        if old_anchors and anchor_source == 1:
            if not path.exists():
                return {"i": i, "ok": False, "reason": "missing_file", "timings": timings}
            t0 = time.perf_counter()
            with fits.open(path, memmap=True, ignore_missing_simple=True, ignore_missing_end=True) as hdul:
                timings["open"] = time.perf_counter() - t0
                t0 = time.perf_counter()
                img_data = hdul[0].data.copy()
                timings["read"] = time.perf_counter() - t0
        else:
            if not path.exists():
                return {"i": i, "ok": False, "reason": "missing_dir", "timings": timings}
            prefix = path.name.strip()
            base = f"SGA2025_{prefix}"
            bands = {}
            t0 = time.perf_counter()
            for band in ("g", "r", "z"):
                p = path / f"{base}-image-{band}.fits.fz"
                try:
                    bands[band] = fitsio.read(p)[39:191, 39:191]
                except OSError:
                    continue
            if len(bands) < 3:
                p = path / f"{base}-image-i.fits.fz"
                try:
                    bands["i"] = fitsio.read(p)[39:191, 39:191]
                except OSError:
                    pass
            timings["read"] = time.perf_counter() - t0
 
            if len(bands) < 1:
                return {"i": i, "ok": False, "reason": "no_bands", "timings": timings}
 
            t0 = time.perf_counter()
            g = get_band(bands, ["g", "r", "i", "z"])
            r = get_band(bands, ["r", "i", "g", "z"])
            z = get_band(bands, ["z", "i", "r", "g"])
            if any(b is None for b in [g, r, z]) or not all(b.shape == (npix, npix) for b in [g, r, z]):
                return {"i": i, "ok": False, "reason": "bad_shape", "timings": timings}
            img_data = np.stack([g, r, z], axis=0).astype(np.float32, copy=False)
            timings["stack"] = time.perf_counter() - t0
 
    except Exception as e:
        return {"i": i, "ok": False, "reason": f"error:{e}", "timings": timings}
 
    timings["total"] = time.perf_counter() - t_start
    result = {"i": i, "ok": True, "timings": timings}
    if return_nbytes:
        # Report size instead of shipping the array back -- ships an array
        # through IPC (ProcessPoolExecutor) too, which would conflate real
        # read speed with pickling/IPC overhead you won't have in the real
        # pipeline (real pipeline stacks images inside the worker's own
        # process/thread, same as here -- it just also writes to HDF5).
        result["nbytes"] = img_data.nbytes
    return result
 
 
def build_sample(catalog, sample_size=500, seed=0, exclude_anchors=True):
    """
    Random sample from the catalog for benchmarking read throughput.
 
    exclude_anchors=True (default) drops anchor_source==1 rows before
    sampling. The old_anchors read path (a single pre-cropped FITS file)
    is structurally much faster than the real per-galaxy path (opening
    up to 4 tiled .fits.fz files, decompressing, and falling back across
    bands), so mixing anchors in would bias throughput numbers upward
    and give you a rosier ETA than the actual ~500k-galaxy run will see.
    Set to False only if you specifically want to benchmark the anchor
    path itself.
    """
    if exclude_anchors:
        catalog = catalog[catalog["anchor_source"] != 1]
        if len(catalog) == 0:
            raise ValueError("No non-anchor rows left to sample -- catalog is all anchors.")
 
    rng = np.random.default_rng(seed)
    n = min(sample_size, len(catalog))
    idx = rng.choice(catalog.index.to_numpy(), size=n, replace=False)
    return catalog.loc[idx].reset_index(drop=True)

In [3]:
def run_benchmark(catalog, npix=152, old_anchors=True, sample_size=500,
                   thread_worker_counts=(4, 8, 16, 32, 64),
                   process_worker_counts=(4, 8, 16, 32),
                   seed=0, exclude_anchors=True, disjoint=False):
    """
    Runs read_one_bench across the sample for each (executor, workers)
    combo and reports throughput + a read-time breakdown.
 
    exclude_anchors=True (default) benchmarks only the non-anchor
    galaxies -- see build_sample() docstring for why.
 
    disjoint=False (default): every config reads the *same* sample_size
    rows. Gives each config maximum statistical power, but after the
    first config runs, those files may still be warm in the OS/Lustre
    client cache, so later configs (or the ones with more workers,
    depending on run order) can look artificially fast. This matters
    most when your sample is small relative to available RAM -- e.g.
    testing against a small catalog file with only ~1000 rows total,
    which likely fits entirely in cache after one pass.
 
    disjoint=True: splits the sample into non-overlapping chunks, one
    per (executor, worker) config, so no file is read twice across the
    whole benchmark run. Removes the caching bias entirely, at the cost
    of a smaller (noisier) sample per config. Recommended when your
    available non-anchor catalog is only a few hundred to ~1000 rows.
 
    Returns a pandas DataFrame you can sort/plot.
    """
    configs = [(ThreadPoolExecutor, w) for w in thread_worker_counts] + \
              [(ProcessPoolExecutor, w) for w in process_worker_counts]
 
    if disjoint:
        full_sample = build_sample(catalog, sample_size=sample_size, seed=seed,
                                    exclude_anchors=exclude_anchors)
        chunk_size = len(full_sample) // len(configs)
        if chunk_size < 30:
            print(f"WARNING: only {chunk_size} galaxies per config with disjoint=True -- "
                  f"timing percentiles will be noisy. Consider testing fewer configs "
                  f"(shorter thread_worker_counts/process_worker_counts) or disjoint=False.\n")
        samples = [full_sample.iloc[i * chunk_size:(i + 1) * chunk_size]
                   for i in range(len(configs))]
    else:
        shared_sample = build_sample(catalog, sample_size=sample_size, seed=seed,
                                      exclude_anchors=exclude_anchors)
        samples = [shared_sample] * len(configs)
 
    def make_args(sample):
        return [
            (row.Index, str(row.Path).strip(), int(row.anchor_source), old_anchors, npix, True)
            for row in sample.itertuples()
        ]
 
    def run_one_config(executor_cls, n_workers, sample):
        args_list = make_args(sample)
        t0 = time.perf_counter()
        rows = []
        with executor_cls(max_workers=n_workers) as ex:
            futures = [ex.submit(read_one_bench, a) for a in args_list]
            for fut in as_completed(futures):
                rows.append(fut.result())
        wall = time.perf_counter() - t0
 
        ok = [r for r in rows if r["ok"]]
        failed = len(rows) - len(ok)
        throughput = len(ok) / wall if wall > 0 else float("nan")
 
        if ok:
            read_times = np.array([r["timings"]["read"] for r in ok])
            open_times = np.array([r["timings"]["open"] for r in ok])
            total_times = np.array([r["timings"]["total"] for r in ok])
        else:
            read_times = open_times = total_times = np.array([0.0])
 
        return {
            "executor": executor_cls.__name__,
            "workers": n_workers,
            "n_sample": len(args_list),
            "n_failed": failed,
            "wall_s": round(wall, 3),
            "galaxies_per_sec": round(throughput, 2),
            "mean_read_ms": round(read_times.mean() * 1000, 2),
            "p95_read_ms": round(np.percentile(read_times, 95) * 1000, 2),
            "mean_open_ms": round(open_times.mean() * 1000, 2),
            "mean_total_ms": round(total_times.mean() * 1000, 2),
        }
 
    total_n = sum(len(s) for s in samples) if disjoint else len(samples[0])
    print(f"Benchmarking (disjoint={disjoint}), total galaxies touched={total_n}\n")
 
    results_summary = []
    for (executor_cls, w), sample in zip(configs, samples):
        r = run_one_config(executor_cls, w, sample)
        results_summary.append(r)
        label = "Thread " if executor_cls is ThreadPoolExecutor else "Process"
        print(f"[{label} x{w:>3}] {r['galaxies_per_sec']:>7.1f} gal/s | "
              f"wall={r['wall_s']:>6.2f}s | mean_read={r['mean_read_ms']:>6.1f}ms | "
              f"n={r['n_sample']} | failed={r['n_failed']}")
 
    df = pd.DataFrame(results_summary)
    best = df.loc[df["galaxies_per_sec"].idxmax()]
    est_hours_500k = 500_000 / best["galaxies_per_sec"] / 3600
    print(f"\nBest config: {best['executor']} x{best['workers']} -> "
          f"{best['galaxies_per_sec']:.1f} gal/s "
          f"(~{est_hours_500k:.2f} hours for 500k galaxies, read-only estimate, "
          f"does not include HDF5 write time)")
 
    return df

In [4]:
def format_data(data, anchors, data_columns, anchor_columns):
    df = read_catalog(data)
    anchor_df = read_catalog(anchors)
    df = (df[list(data_columns.keys())].rename(columns=data_columns))
    df["anchor_source"] = 0
    anchor_df = (anchor_df[list(anchor_columns.keys())].rename(columns=anchor_columns))
    anchor_df["anchor_source"] = 1
    return pd.concat([df, anchor_df], ignore_index=True)

def read_catalog(path): # Read in original data or anchors
    path = Path(path)

    if path.suffix == ".csv":
        return pd.read_csv(path)

    if path.suffix == ".fits":
        return Table.read(path, hdu=1).to_pandas()

    raise ValueError(f"Unsupported file type: {path.suffix}")

In [5]:
data_path = Path("/pscratch/sd/q/qshimp/Sorter/sga2025_sample.csv")   
anchor_path = Path("/pscratch/sd/q/qshimp/SGA2020-data/Anchors/VI_4000_sga152x152_complete.csv")
# IMPORTANT: CHANGE TO MATCH YOUR DATA
data_columns = {
    # Change  -  Don't change
    "target_ra": "target_ra",
    "target_dec":"target_dec",
    "g_mag":     "g_mag",
    "r_mag":     "r_mag",
    "z_mag":     "z_mag",
    "Main_Type": "Main_Type",
    "targetid":  "targetid",
    "Path":      "Path"
}
anchor_columns = {
    # Change  -  Don't change
    "ra":       "target_ra",
    "dec":      "target_dec",
    "g_mag":    "g_mag",
    "r_mag":    "r_mag",
    "z_mag":    "z_mag",
    "Main_type":"Main_Type",
    "ref_id":   "targetid",
    "Path":     "Path"
}
OLD_ANCHORS=True
targets_sample = format_data(data_path, anchor_path, data_columns, anchor_columns)
results_df = run_benchmark(
    targets_sample,
    npix=152,
    old_anchors=OLD_ANCHORS,
    sample_size=1000,               # use everything you've got
    thread_worker_counts=(8, 16),
    process_worker_counts=(8, 16, 32, 64),
    disjoint=True,                  # avoids the cache bias
)
results_df.sort_values("galaxies_per_sec", ascending=False)

Benchmarking (disjoint=True), total galaxies touched=996

[Thread  x  8]   105.1 gal/s | wall=  1.58s | mean_read=  36.5ms | n=166 | failed=0
[Thread  x 16]   118.0 gal/s | wall=  1.41s | mean_read=  50.1ms | n=166 | failed=0
[Process x  8]   522.7 gal/s | wall=  0.32s | mean_read=   9.3ms | n=166 | failed=0
[Process x 16]   528.3 gal/s | wall=  0.31s | mean_read=  10.5ms | n=166 | failed=0
[Process x 32]   368.2 gal/s | wall=  0.45s | mean_read=  13.1ms | n=166 | failed=0
[Process x 64]   211.1 gal/s | wall=  0.79s | mean_read=  21.1ms | n=166 | failed=0

Best config: ProcessPoolExecutor x16 -> 528.3 gal/s (~0.26 hours for 500k galaxies, read-only estimate, does not include HDF5 write time)


,executor,workers,n_sample,n_failed,wall_s,galaxies_per_sec,mean_read_ms,p95_read_ms,mean_open_ms,mean_total_ms
3,ProcessPoolExecutor,16,166,0,0.314,528.29,10.48,19.36,0.0,10.63
2,ProcessPoolExecutor,8,166,0,0.318,522.73,9.30,22.90,0.0,9.42
4,ProcessPoolExecutor,32,166,0,0.451,368.21,13.13,20.33,0.0,13.30
5,ProcessPoolExecutor,64,166,0,0.786,211.07,21.13,27.57,0.0,21.37
1,ThreadPoolExecutor,16,166,0,1.406,118.05,50.08,117.95,0.0,112.34
0,ThreadPoolExecutor,8,166,0,1.580,105.06,36.55,83.04,0.0,73.55
